MÁY hút bụi Simple

In [ ]:
import random
def P_moves(x, y):
    moves = []
    if (x < 3): 
        moves.append('D')
    if (x > 0): 
        moves.append('U')
    if (y < 3): 
        moves.append('R')
    if (y > 0): moves.append('L')
    return moves

def Matrix_Random(rows, cols):
    matrix = [[random.randint(0,1) for _ in range(cols)] for _ in range(rows)]
    return matrix

def Thuchien(action, matrix, x, y):
    if action == 'D':
        x += 1
    elif action == 'U':
        x -= 1
    elif action == 'R':
        y += 1
    elif action == 'L':
        y -= 1

    return matrix[x][y], x, y

def is_full_zero(matrix): # kiểm tra nhà sạch chưa 
    return sum(sum(row) for row in matrix) == 0

# khai báo ban đầu
m_matrix = Matrix_Random(4, 4)
i, j = 0, 0
state = m_matrix[i][j]

# chạy chương trình
while (True):
    moves = P_moves(i, j) # các chiều có thế chọn
    action = random.choice(moves) #chọn chiều ngẫu nhiên trong moves
    if state == 1: 
        m_matrix[i][j] = 0
    state, i, j = Thuchien(action, m_matrix, i, j) #thực hiện chiều đã trọn ở trên
    print(m_matrix)

    #kiểm tra mảng full 0 chưa
    if is_full_zero(m_matrix): 
        print("successfully")
        break
            

8 puzzle model

In [ ]:
import random
def P_moves(x, y):
    moves = []
    if (x < 2): 
        moves.append('D')
    if (x > 0): 
        moves.append('U')
    if (y < 2): 
        moves.append('R')
    if (y > 0): moves.append('L')
    return moves

# hàm tạo mảng ngẫu nhiên ban đầu
def matrix_random():
    numbers = list(range(9))  # [0,1,2,3,4,5,6,7,8]
    random.shuffle(numbers)   # đảo ngẫu nhiên
    return [numbers[i:i+3] for i in range(0, 9, 3)]

#tìm vị trí trống
def find_zero(matrix):
    for i in range(3):
        for j in range(3):
            if matrix[i][j] == 0:
                return i, j
            
# cập nhật trạng thái sau hành động
def update_state(state, action):
    x, y = find_zero(state)

    new_state = [row[:] for row in state]

    if action == "U":
        new_state[x][y], new_state[x - 1][y] = new_state[x - 1][y], new_state[x][y]

    elif action == "D":
        new_state[x][y], new_state[x + 1][y] = new_state[x + 1][y], new_state[x][y]

    elif action == "L":
        new_state[x][y], new_state[x][y - 1] = new_state[x][y - 1], new_state[x][y]

    elif action == "R":
        new_state[x][y], new_state[x][y + 1] = new_state[x][y + 1], new_state[x][y]

    return new_state

# Hành động ngược lại
# để tránh đi tới rồi quay lại ngay
def opposite(action):
    opposite_moves = {
        "U": "D",
        "D": "U",
        "L": "R",
        "R": "L"
    }
    return opposite_moves.get(action)


# RULE-MATCH
# chọn random action
# nhưng loại bỏ action ngược
# với hành động vừa thực hiện
def rule_match(state, last_action):
    x, y = find_zero(state)
    possible = P_moves(x, y)

    # loại bỏ hành động ngược với bước trước
    if last_action:
        reverse = opposite(last_action)
        if reverse in possible and len(possible) > 1:
            possible.remove(reverse)

    return random.choice(possible)

# In ma trận
def print_matrix(matrix):
    for row in matrix:
        print(row)
    print()

goal = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

# khởi tạo mảng random
state = matrix_random()

print("Trạng thái ban đầu:")
print_matrix(state)

step = 0
max_step = 50
last_action = None

check = True
while state != goal and step < max_step:
    # percept = trạng thái hiện tại
    percept = state

    # chọn action
    action = rule_match(percept, last_action)

    print(f"Bước {step + 1}: Action = {action}")

    # cập nhật state
    state = update_state(state, action)

    print_matrix(state)

    # lưu action vừa thực hiện
    last_action = action
    step += 1

if state == goal:
    print("Đã đạt trạng thái đích!")
else:
    print("Chưa đạt trạng thái đích sau", max_step, "bước.")

Máy hút bụi model based reflex agent

In [ ]:
import random

# xác định các hướng có thể đi từ vị trí (x, y)
def P_moves(x, y, last_action):
    moves = []

    if x < 3:
        moves.append('D')   # xuống
    if x > 0:
        moves.append('U')   # lên
    if y < 3:
        moves.append('R')   # phải
    if y > 0:
        moves.append('L')   # trái

    # loại bỏ hướng ngược lại với bước vừa đi
    opposite = {
        'D': 'U',
        'U': 'D',
        'R': 'L',
        'L': 'R'
    }

    if last_action in opposite:
        reverse_move = opposite[last_action]
        if reverse_move in moves:
            moves.remove(reverse_move)

    return moves


def Matrix_Random(rows, cols):
    return [[random.randint(0, 1) for _ in range(cols)] for _ in range(rows)]


def Update_State(state, action, model):
    x, y = state

    if action == 'D':
        x += 1
    elif action == 'U':
        x -= 1
    elif action == 'R':
        y += 1
    elif action == 'L':
        y -= 1

    percept = model[x][y]
    return (x, y), percept


def Rule_Match(state, percept, last_action):
    x, y = state

    
    if percept == 1:
        return "CLEAN"

    # nếu sạch thì chọn hướng ngẫu nhiên nhưng không quay đầu ngay
    moves = P_moves(x, y, last_action)
    return random.choice(moves)


def is_full_zero(matrix):
    return sum(sum(row) for row in matrix) == 0


model = Matrix_Random(4, 4)
state = (0, 0)
action = None
last_move = None
percept = model[0][0]

print("Ma trận ban đầu:")
for row in model:
    print(row)

print("\nBắt đầu chạy Agent:\n")

while True:

    # cập nhật trạng thái nếu hành động trước là di chuyển
    if action is not None and action != "CLEAN":
        state, percept = Update_State(state, action, model)
        last_move = action

    # chọn luật
    action = Rule_Match(state, percept, last_move)

    x, y = state

    # thực hiện hành động
    if action == "CLEAN":
        model[x][y] = 0
        percept = 0
        print(f"Hút bụi tại ({x}, {y})")

    else:
        print(f"Di chuyển {action} từ ({x}, {y})")

    for row in model:
        print(row)
    print()

    if is_full_zero(model):
        print("Successfully! Nhà đã sạch hoàn toàn.")
        break